# Project Part 1: Exploring CLIP for Zero-Shot Learning and Probing

**Total Marks: 20**

**Group Name:** __________________

**Student Name (Student ID):**

1. __________________ (__________________)
2. __________________ (__________________)
3. __________________ (__________________)
4. __________________ (__________________)

## Introduction

This assignment deepens your understanding of large-scale vision-language models such as CLIP (Contrastive Language-Image Pre-Training). You will apply CLIP to image classification in two ways:

1. **Zero-Shot Prediction**: classify images by comparing them with arbitrary text descriptions, with no task-specific training. You will implement the prediction logic and investigate how prompts and similarity metrics affect performance.
2. **Probing a frozen backbone**: use CLIP's image encoder as a fixed feature extractor and **build and train** a small classification head on top of its features.

You will compare the performance, trade-offs and characteristics of the two approaches on ImageNet.

**Marks:** Q1 Setting up CLIP [3] · Q2 Zero-shot analysis [6] · Q3 Probing [6] · Q4 Paper reading [5]

**How to answer:** write all written answers in the **"Your answer"** Markdown cells in this notebook. Run the notebook from top to bottom before submitting so that all outputs are saved.

**Tested with:** Python 3.12, `torch` ≥ 2.4, `transformers` ≥ 5.0 (also works on 4.4x), `datasets` ≥ 3.0, `scikit-learn` ≥ 1.4.

---

## **Question 1: Setting up CLIP** [3 Marks]

### **Step 1: Environment Setup and Model Loading (Provided)**

This section loads the pre-trained CLIP model and processor so that you can focus on the core logic.

> You may see a `LOAD REPORT` listing `position_ids` as `UNEXPECTED`, and a note about the *fast* image processor. Both are harmless.

In [ ]:
import os
import json
import random
from pathlib import Path
from typing import Optional, Tuple, List, Dict

import numpy as np
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torch.utils.data import Dataset as TorchDataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score as sk_accuracy_score, f1_score as sk_f1_score

import datasets
from datasets import Features, Value
try:                                   # datasets >= 4.0
    from datasets import List as HFList
except ImportError:                    # older datasets
    from datasets import Sequence as HFList
from transformers import CLIPModel, CLIPProcessor

# Reproducibility
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

CACHE_DIR = "./cache"  # Cache directory for the pre-computed CLIP image features

model_str = "openai/clip-vit-base-patch32"
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()
                      else "cpu")

# Load CLIP with PyTorch's scaled-dot-product attention for efficiency
model = CLIPModel.from_pretrained(model_str, attn_implementation="sdpa").to(device)
model.eval()  # evaluation mode: CLIP is never trained in Part 1

# The processor prepares images (resize, crop, normalise) and text (tokenise)
processor = CLIPProcessor.from_pretrained(model_str)

print(f"Model loaded on device: {device}")
print(f"Embedding dimension (projection_dim): {model.config.projection_dim}")

### **Step 2: A Quick Demonstration (Provided)**

Load a demonstration image from the web to show a simple example of what CLIP can do.

In [ ]:
demo_image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
print(f"Loading a demo image from: {demo_image_url}")
demo_image = Image.open(requests.get(demo_image_url, stream=True, timeout=30).raw).convert("RGB")
demo_image

### **Step 3: Implementing the Core CLIP Encoder** [3 Marks]

**Your Task:** Implement `encode_image` [1], `encode_text` [1] and `predict` [1].

**Version note (important).** In `transformers` ≥ 5.0, `get_image_features` / `get_text_features` return a *model output object*, not a tensor; the projected embedding is stored in its `.pooler_output` field. In `transformers` 4.x they return the tensor directly. The helper `_as_embedding` (provided) handles both cases. Use it on the result of `get_image_features` / `get_text_features`.

**About `logit_scale`.** CLIP was trained with similarities multiplied by a learned temperature `logit_scale.exp()` (≈ 100). Without it, the softmax over cosine similarities (which lie in [-1, 1]) is almost uniform.

In [ ]:
class CLIPEncoder:
    """
    Wraps CLIP's encoding functions for images and text, plus zero-shot prediction.
    """
    def __init__(self, model: CLIPModel, processor: CLIPProcessor, device: torch.device):
        self.model = model
        self.processor = processor
        self.device = device

    @staticmethod
    def _as_embedding(output) -> torch.Tensor:
        """(Provided) Return the projected embedding as a tensor in any transformers version."""
        if isinstance(output, torch.Tensor):      # transformers 4.x
            return output
        return output.pooler_output               # transformers >= 5.0

    @torch.no_grad()
    def encode_image(self, images: List[Image.Image]) -> torch.Tensor:
        """
        Encode a batch of PIL images into CLIP embeddings.

        Instructions:
        1. Use `self.processor(images=..., return_tensors="pt")` to preprocess the images.
        2. Move every tensor in the processed inputs to `self.device`.
        3. Call `self.model.get_image_features(**inputs)` and pass the result through
           `self._as_embedding(...)`.
        4. Do NOT normalise here (normalisation is decided in `predict`).

        Returns:
            torch.Tensor of shape (B, 512): one un-normalised embedding per image.

        Docs: https://huggingface.co/docs/transformers/model_doc/clip
        """

        # ================================
        # TODO: YOUR CODE GOES HERE (1 Mark)
        # ================================
        raise NotImplementedError("Please implement `encode_image`.")

    @torch.no_grad()
    def encode_text(self, text: List[str]) -> torch.Tensor:
        """
        Encode a list of strings into CLIP embeddings.

        Instructions:
        1. Use `self.processor(text=..., return_tensors="pt", padding=True, truncation=True)`.
           (CLIP's text encoder accepts at most 77 tokens; longer text is truncated.)
        2. Move every tensor in the processed inputs to `self.device`.
        3. Call `self.model.get_text_features(**inputs)` and pass the result through
           `self._as_embedding(...)`.
        4. Do NOT normalise here.

        Returns:
            torch.Tensor of shape (T, 512): one un-normalised embedding per string.
        """

        # ================================
        # TODO: YOUR CODE GOES HERE (1 Mark)
        # ================================
        raise NotImplementedError("Please implement `encode_text`.")

    @torch.no_grad()
    def predict(self, image_features: torch.Tensor, text_features: torch.Tensor,
                criterion: str) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Zero-shot prediction: score every image against every text and pick the best text.

        Args:
            image_features: (N, D) image embeddings (from `encode_image`).
            text_features:  (T, D) text embeddings (from `encode_text`), one per class.
            criterion:      "dot_product"       -> score = image · text (raw embeddings)
                            "cosine_similarity" -> L2-normalise both (F.normalize, dim=-1),
                                                   then take the dot product.

        Instructions:
        1. Make sure both tensors are on the same device and dtype (float32).
        2. If criterion == "cosine_similarity", L2-normalise both sets of features.
        3. Compute the (N, T) score matrix.
        4. Multiply the scores by CLIP's temperature `self.model.logit_scale.exp()`
           to obtain logits, then apply softmax over the T classes.
        5. The prediction is the argmax over classes.

        Returns:
            predicted_idx: LongTensor of shape (N,)   - index of the best text for each image
            probs:         FloatTensor of shape (N, T) - softmax probabilities (rows sum to 1)
        """
        assert criterion in ["dot_product", "cosine_similarity"], "Invalid criterion"

        # ================================
        # TODO: YOUR CODE GOES HERE (1 Mark)
        # ================================
        raise NotImplementedError("Please implement `predict`.")


# Initialise the encoder (this uses your implementation)
encoder = CLIPEncoder(model, processor, device)

### **Step 4: Test Case for the Encoder (Provided)**

A quick check on the demo image (two cats on a sofa). Expected: the prediction is `'a photo of two cats'` with a clearly dominant probability.

In [ ]:
try:
    demo_labels = ["a photo of a cat", "a photo of a dog", "a photo of two cats"]
    image_features = encoder.encode_image([demo_image])
    text_features = encoder.encode_text(demo_labels)

    assert isinstance(image_features, torch.Tensor), "encode_image must return a torch.Tensor"
    assert image_features.shape == (1, 512), f"expected (1, 512), got {tuple(image_features.shape)}"
    assert text_features.shape == (3, 512), f"expected (3, 512), got {tuple(text_features.shape)}"

    predicted_idx, probs = encoder.predict(image_features, text_features, "cosine_similarity")
    assert predicted_idx.shape == (1,) and probs.shape == (1, 3), "check the shapes returned by predict"
    assert torch.allclose(probs.sum(dim=-1).cpu(), torch.ones(1)), "probabilities must sum to 1"

    print(f"Predicted label: '{demo_labels[predicted_idx.item()]}'")
    for label, p in zip(demo_labels, probs[0].tolist()):
        print(f"  {p:.4f}  {label}")
    print("\n✅ Encoder checks passed.")
except NotImplementedError:
    print("One or more methods in CLIPEncoder are not yet implemented.")

### **Step 5: Data Loading and Preparation (Provided)**

This step streams the ImageNet-1k **validation** split (50,000 images, 1,000 classes), encodes every image once with your `encode_image`, and caches the 512-d features in `./cache`. Later runs load the cache instantly.

> ⏱ **Run time:** encoding is a one-off cost. It can take an hour or more on a CPU-only laptop. Leave it running, and do not delete `./cache` afterwards. If you change `encode_image`, delete `./cache/imagenet_val_clip` so the features are rebuilt.
>
> 📄 The file `id_to_label.json` (class index → class name) is provided with this notebook and must be in the same folder.

We split the 50,000 features into three stratified sets (35 / 5 / 10 images per class):

| Set | Size | Used for |
|---|---|---|
| Train | 35,000 | training the probe (Q3) |
| Validation | 5,000 | every **choice**: the best prompt (Q2) and the best training epoch (Q3) |
| Test | 10,000 | the **final** numbers of both methods, computed once after all choices are made |

Choosing the prompt or the epoch by test accuracy and then reporting that same test accuracy would make the result optimistic, because the test set would have influenced the model. Note this is not the standard ImageNet benchmark protocol, so numbers differ from the CLIP paper.

In [ ]:
def load_imagenet_val_clip_features(
    encoder: "CLIPEncoder",
    cache_path: str = os.path.join(CACHE_DIR, "imagenet_val_clip"),
    batch_size: int = 128,
    tqdm_total: Optional[int] = 50000,  # ImageNet-1k val has 50,000 images
) -> datasets.Dataset:
    """
    Stream the ImageNet validation split, encode images into CLIP vectors,
    cache features + labels to disk, and return a Dataset that yields torch tensors.
    """
    if os.path.exists(cache_path):
        print("Loading CLIP features from local cache...")
        ds = datasets.load_from_disk(cache_path)
        return ds.with_format("torch", columns=["features", "label"])

    print("Downloading ImageNet validation (streaming mode)...")
    stream_dataset = datasets.load_dataset(
        "benjamin-paine/imagenet-1k-256x256", split="validation", streaming=True
    )

    feats_chunks: List[torch.Tensor] = []
    labels_all: List[int] = []
    buf_images, buf_labels = [], []

    def flush():
        nonlocal buf_images, buf_labels
        if not buf_images:
            return
        feats = encoder.encode_image(buf_images).to("cpu", dtype=torch.float32)
        feats_chunks.append(feats)
        labels_all.extend(buf_labels)
        buf_images, buf_labels = [], []

    print("Encoding ImageNet validation set with CLIP...")
    for row in tqdm(stream_dataset, desc="Encoding", total=tqdm_total):
        buf_images.append(row["image"].convert("RGB"))
        buf_labels.append(int(row["label"]))
        if len(buf_images) >= batch_size:
            flush()
    flush()  # remaining images

    all_features = torch.cat(feats_chunks, dim=0).numpy().astype(np.float32)
    labels_np = np.asarray(labels_all, dtype=np.int64)

    schema = Features({
        "features": HFList(Value("float32"), length=all_features.shape[1]),
        "label": Value("int64"),
    })
    ds = datasets.Dataset.from_dict({"features": all_features, "label": labels_np}, features=schema)
    ds.save_to_disk(cache_path)
    print("CLIP features cached locally!")
    return ds.with_format("torch", columns=["features", "label"])


dataset = load_imagenet_val_clip_features(encoder)

# Class names, ordered by class index 0..999
with Path("./id_to_label.json").open("r") as f:
    idx_to_label = json.load(f)
class_labels = [name for _, name in sorted(idx_to_label.items(), key=lambda kv: int(kv[0]))]
assert len(class_labels) == 1000, f"expected 1000 classes, got {len(class_labels)}"

all_features = torch.stack([item["features"] for item in dataset]).float()
all_labels = torch.tensor([int(item["label"]) for item in dataset], dtype=torch.int64)

# 1) Hold out 10,000 test images (10 per class)
trainval_images, test_images, trainval_labels, test_labels = train_test_split(
    all_features, all_labels, test_size=0.2, shuffle=True, random_state=42, stratify=all_labels.numpy()
)
# 2) Take 5,000 of the remaining 40,000 as a validation set (5 per class)
train_images, val_images, train_labels, val_labels = train_test_split(
    trainval_images, trainval_labels, test_size=5000, shuffle=True, random_state=42, stratify=trainval_labels.numpy()
)
train_images, val_images, test_images = train_images.to(device), val_images.to(device), test_images.to(device)
train_labels, val_labels, test_labels = train_labels.to(device), val_labels.to(device), test_labels.to(device)

print(f"Train set shape:      {tuple(train_images.shape)}")
print(f"Validation set shape: {tuple(val_images.shape)}")
print(f"Test set shape:       {tuple(test_images.shape)}")

### **Evaluation Metrics (Provided)**

We use scikit-learn's built-in metrics: accuracy and **macro-averaged** F1, which averages the per-class F1 scores so that every class counts equally.

In [ ]:
class Metrics:
    """Thin wrappers around scikit-learn metrics that accept lists or (GPU) tensors."""
    @staticmethod
    def _to_numpy(x) -> np.ndarray:
        if isinstance(x, torch.Tensor):
            return x.detach().cpu().numpy()
        return np.asarray(x)

    @staticmethod
    def accuracy_score(predictions, targets) -> float:
        return float(sk_accuracy_score(Metrics._to_numpy(targets), Metrics._to_numpy(predictions)))

    @staticmethod
    def f1_score(predictions, targets) -> float:
        return float(sk_f1_score(Metrics._to_numpy(targets), Metrics._to_numpy(predictions),
                                 average="macro", zero_division=0))

metrics = Metrics()
print("Example:", metrics.accuracy_score([0, 1, 2, 2], [0, 1, 1, 2]), metrics.f1_score([0, 1, 2, 2], [0, 1, 1, 2]))

---

## **Question 2: Zero-Shot Classification Analysis** [6 Marks]

### **Step 6: Prompt Engineering** [2 Marks]

**Your Task:** Complete the loop in `evaluate_zero_shot_prompts`. For each template:

1. Build one prompt per class with `template.format(class_name)`.
2. Encode the prompts with `encoder.encode_text`.
3. Predict on `images` (the **validation** features passed in) with `encoder.predict(..., "cosine_similarity")`.
4. Store the accuracy against `labels` (use `metrics.accuracy_score`) in `results[template]`.

The prompts are compared on the validation set because Step 8 uses the result to **choose** a prompt.

Then **add at least two prompt templates of your own** to `prompts_to_test`.

In [ ]:
def evaluate_zero_shot_prompts(
    prompt_templates: List[str],
    class_labels: List[str],
    images: torch.Tensor,
    labels: torch.Tensor,
    encoder: "CLIPEncoder",
    metrics: "Metrics",
) -> Dict[str, float]:
    results = {}
    print("Evaluating different prompts...")
    for template in tqdm(prompt_templates, desc="Prompts"):
        # ================================
        # TODO: YOUR CODE GOES HERE (2 Marks)
        # ================================
        raise NotImplementedError("Please implement the prompt evaluation loop.")
    return results

prompts_to_test = [
    "a photo of a {}",
    "a picture of a {}",
    "an image of a {}",
    "{}",
    "a wild animal: {}",
    "Beneath the fading sunset, the curious child wandered along the winding path. This is an image of {}.",
    # TODO: add at least two prompt templates of your own below
]

prompt_accuracies = evaluate_zero_shot_prompts(prompts_to_test, class_labels, val_images, val_labels, encoder, metrics)
print("Validation accuracy per prompt:")
for prompt, accuracy in sorted(prompt_accuracies.items(), key=lambda kv: -kv[1]):
    print(f'{accuracy:.4f}  "{prompt}"')

### **Step 7: Analysis of Prompt Engineering** [2 Marks]

1. Record the validation accuracy of every prompt you tried (a table is fine). Which works best, and why? [1]
2. What happens when unrelated text is added in front of the class name (the "Beneath the fading sunset…" prompt)? Explain why. [1]

**Your answer:**

_(write your answer here)_

### **Step 8: Similarity Metric Comparison** [1 Mark]

**Your Task:**

1. Select the best prompt from Step 6 (choose it programmatically from the **validation** results in `prompt_accuracies`).
2. Build and encode the corresponding class prompts.
3. Predict on the **test** set (`test_images`) with `"dot_product"` and compute the accuracy.
4. Predict on the test set with `"cosine_similarity"` and compute the accuracy. This is the final zero-shot test accuracy used in Step 12.
5. Inspect the L2 norms of the image and text embeddings (provided code below).

In [ ]:
best_prompt, accuracy_dot, accuracy_cos = None, None, None
text_features = None
# ================================
# TODO: YOUR CODE GOES HERE (1 Mark)
# ================================


if accuracy_dot is not None and accuracy_cos is not None:
    print(f"Using prompt: '{best_prompt}'\n")
    print(f"Accuracy with dot_product:       {accuracy_dot:.4f}")
    print(f"Accuracy with cosine_similarity: {accuracy_cos:.4f}")

    # (Provided) Look at the embedding norms — useful for Step 9
    img_norms, txt_norms = test_images.norm(dim=-1), text_features.norm(dim=-1)
    print(f"\nImage-embedding norms: min={img_norms.min():.2f}  max={img_norms.max():.2f}")
    print(f"Text-embedding norms:  min={txt_norms.min():.2f}  max={txt_norms.max():.2f}")
else:
    print("Step 8 not implemented yet.")

### **Step 9: Analysis of the Similarity Metric** [1 Mark]

Does using cosine similarity differ from using the dot product as the decision criterion? Report both accuracies. When do the two measures rank classes identically, and what changes when the embeddings are not L2-normalised? *Hint: for a single image, which vector lengths can change the ranking of the 1,000 classes, and which cannot?*

**Your answer:**

_(write your answer here)_

---

## **Question 3: Probing a Frozen CLIP Backbone** [6 Marks]

**Probing** means training a small classification head on top of a **frozen** encoder: CLIP is not updated, only the head is trained. A strict **linear probe** (as in the CLIP paper) is a single `nn.Linear` layer, i.e. multinomial logistic regression on the features. Here we use a slightly larger head: one residual MLP block followed by a linear output layer. Because the features are pre-computed in Step 5, training is fast even on a CPU.

(In Part 2 you will go further and update CLIP's own weights through parameter-efficient fine-tuning.)

### **Step 10: Implementing the Classification Network** [1 Mark]

**Your Task:** Complete `ClsNetwork`:

- `__init__`: after the provided `self.res_blk`, add a `nn.LayerNorm(input_dim)` and a final `nn.Linear(input_dim, num_classes)`.
- `forward`: residual block → LayerNorm → linear layer. Return the **logits** (no softmax: `nn.CrossEntropyLoss` applies log-softmax itself).

In [ ]:
class ResidualFCNBlock(nn.Module):  # Provided
    def __init__(self, input_dim: int, hidden_dim: int, p: float = 0.2):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.ff = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(p),
            nn.Linear(hidden_dim, input_dim),
            nn.Dropout(p),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.ff(self.norm(x))  # pre-norm residual connection


class ClsNetwork(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        hidden_dim = 4 * input_dim
        self.res_blk = ResidualFCNBlock(input_dim, hidden_dim, p=0.2)
        # ================================
        # TODO: YOUR CODE GOES HERE
        # ================================
        raise NotImplementedError("Please define the layers in ClsNetwork `__init__`.")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # ================================
        # TODO: YOUR CODE GOES HERE
        # ================================
        raise NotImplementedError("Please implement `ClsNetwork.forward`.")


set_seed(42)
classifier_model = ClsNetwork(input_dim=train_images.shape[1], num_classes=len(class_labels)).to(device)
n_params = sum(p.numel() for p in classifier_model.parameters())
print(f"Trainable parameters in the head: {n_params:,}")

### **Step 11: Implementing the Training Step** [2 Marks]

**Your Task:**

1. [1] Inside `Trainer.train`, implement the core training step: zero the gradients, forward pass, compute `loss` with `criterion`, backward pass, optimiser step.
2. [1] At the bottom of the cell, create a `Trainer` with `train_loader` and `val_loader`, and train for at most **10 epochs** with **lr = 1e-4** and **patience = 3**.

**Early stopping (provided).** The trainer evaluates on the **validation** set after every epoch. If validation accuracy has not improved for `patience` consecutive epochs, training stops. At the end, the weights of the epoch with the best validation accuracy are restored. (`patience=None` always runs all epochs.) The test set is evaluated once, in the cell after training.

In [ ]:
class Trainer:
    def __init__(self, model, train_loader, val_loader, device):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.writer = SummaryWriter("runs/probe_experiment")
        self.history = []
        self.best_acc, self.best_epoch, self.best_state = -1.0, 0, None

    def train(self, epochs: int, lr: float, patience: Optional[int] = None):
        optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        criterion = nn.CrossEntropyLoss()
        epochs_without_improvement = 0

        for epoch in range(epochs):
            self.model.train()
            total_loss = 0.0
            pbar = tqdm(self.train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=False)
            for step, (features, labels) in enumerate(pbar, start=1):
                features, labels = features.to(self.device), labels.to(self.device)
                # ================================
                # TODO: YOUR CODE GOES HERE (1 Mark): the core training step
                # ================================
                raise NotImplementedError("Please implement the core training step.")

                total_loss += loss.item()
                pbar.set_postfix(loss=f"{total_loss / step:.4f}")

            avg_loss = total_loss / len(self.train_loader)
            val_acc, val_f1 = self.evaluate(self.val_loader)
            self.history.append({"epoch": epoch + 1, "train_loss": avg_loss, "val_accuracy": val_acc, "val_f1": val_f1})
            self.writer.add_scalar("Loss/train", avg_loss, epoch)
            self.writer.add_scalar("Accuracy/val", val_acc, epoch)
            self.writer.add_scalar("F1_Score/val", val_f1, epoch)
            print(f"Epoch {epoch + 1}: train_loss={avg_loss:.4f}  val accuracy={val_acc:.4f}  val macro-F1={val_f1:.4f}")
            if val_acc > self.best_acc:  # keep the best epoch on validation
                self.best_acc, self.best_epoch = val_acc, epoch + 1
                self.best_state = {k: v.detach().clone() for k, v in self.model.state_dict().items()}
                epochs_without_improvement = 0
            else:
                epochs_without_improvement += 1
                if patience is not None and epochs_without_improvement >= patience:
                    print(f"Early stopping: no improvement in validation accuracy for {patience} epochs.")
                    break
        self.writer.close()
        self.model.load_state_dict(self.best_state)  # early stopping
        print(f"\nTraining finished! Restored epoch {self.best_epoch} (val accuracy={self.best_acc:.4f}).")

    @torch.no_grad()
    def evaluate(self, loader) -> Tuple[float, float]:
        """Return (accuracy, macro-F1) of the current model on `loader`."""
        self.model.eval()
        all_preds, all_labels = [], []
        for features, labels in loader:
            outputs = self.model(features.to(self.device))
            all_preds.append(outputs.argmax(dim=1).cpu())
            all_labels.append(labels.cpu())
        all_preds, all_labels = torch.cat(all_preds), torch.cat(all_labels)
        return Metrics.accuracy_score(all_preds, all_labels), Metrics.f1_score(all_preds, all_labels)


class ClsDataset(TorchDataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels
    def __len__(self) -> int:
        return len(self.features)
    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.features[idx], self.labels[idx]


train_dataset = ClsDataset(train_images, train_labels)
val_dataset = ClsDataset(val_images, val_labels)
test_dataset = ClsDataset(test_images, test_labels)

g = torch.Generator().manual_seed(42)  # reproducible shuffling
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=g)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

# ================================
# TODO: YOUR CODE (1 Mark): instantiate the Trainer (train_loader, val_loader) and train for at most 10 epochs, lr=1e-4, patience=3
# ================================
raise NotImplementedError("Please instantiate and call the Trainer.")

### **Final test evaluation (Provided)**

All choices are now fixed (prompt chosen on validation in Step 8, epoch chosen on validation by the trainer), so both methods are evaluated **once** on the test set.

In [ ]:
probe_test_acc, probe_test_f1 = trainer.evaluate(test_loader)
print(f"{'Method':<52}{'Test accuracy':>14}{'Test macro-F1':>15}")
print(f"{'Zero-shot (' + repr(best_prompt) + ', cosine)':<52}{accuracy_cos:>14.4f}{'':>15}")
print(f"{'Probe (best validation epoch ' + str(trainer.best_epoch) + ')':<52}{probe_test_acc:>14.4f}{probe_test_f1:>15.4f}")

### **Step 12: Performance Analysis and Comparison** [3 Marks]

Answer in the cell below:

1. [1] Report the **test accuracy and macro-F1** of your trained classifier (printed above, at the best validation epoch), and give a summary table comparing the accuracy of zero-shot classification (best prompt, cosine similarity) and the trained classifier.
2. [1] Analyse the results. Why does the trained classifier outperform the zero-shot approach?
3. [1] Discuss the **trade-offs** between the two methods: performance, data requirement, flexibility, computational cost.

*Tip:* `trainer.history` holds the per-epoch validation metrics, and the cell above prints both final test accuracies; you can also view the curves with `tensorboard --logdir runs`.

**Your answer:**

_(write your answer here)_

---

## **Question 4: Paper Reading** [5 Marks]

Read the paper [*Learning Transferable Visual Models From Natural Language Supervision*](https://arxiv.org/abs/2103.00020) (Radford et al., 2021) and answer the questions below, 1 mark each. Write your answers in the cells provided.

**Q4.1** [1 Mark] What does the CLIP model learn?

**Your answer:**

_(write your answer here)_

**Q4.2** [1 Mark] Explain in at most 3 sentences what "contrastive learning" means.

**Your answer:**

_(write your answer here)_

**Q4.3** [1 Mark] Why do you think CLIP's zero-shot performance can sometimes surpass supervised baselines? What does this say about the generalization abilities of representation learning?

**Your answer:**

_(write your answer here)_

**Q4.4** [1 Mark] How do the labels in CLIP-based zero-shot classification differ from traditional image classification models from supervised learning?

**Your answer:**

_(write your answer here)_

**Q4.5** [1 Mark] What do you think CLIP falls short in, and why do you think this happens?

**Your answer:**

_(write your answer here)_

---

## AI Usage

List every generative AI tool you used for this notebook: which tool, what you asked it (the prompt or a short summary), and which part of the notebook it was used for. Write "None" if you did not use any.

| Tool | What you asked | Where it was used |
|---|---|---|
| | | |